# ML-10 — Content Action Playbook


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The queue reuses the capstone's Gradient Boosting model (`is_declining`, client-grouped
validation, honest precision@K = 0.689). Each flagged row gets one plain-English reason code
from a small, explainable rule set — never an opaque score alone. Alongside the score-based
queue, a lightweight **archetype → action mapping** groups every page by trend direction and
visibility, so even unflagged content has a clear default action.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count", "content_age_days",
    "age_tier_order", "days_since_last_update", "impressions_90d", "clicks_90d", "pageviews_90d",
    "sessions_90d", "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
categorical_features = ["competition_level", "content_type", "main_intent"]
for c in ["word_count", "char_count"]:
    df[f"has_{c}"] = df[c].notna().astype(int)
numeric_features += ["has_word_count", "has_char_count"]

X = df[numeric_features + categorical_features].copy()
y = df["is_declining"]
groups = df["client_id"]

pre = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
gb = Pipeline([("pre", pre), ("clf", GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42))])
gb.fit(X.iloc[train_idx], y.iloc[train_idx])
df["model_score"] = gb.predict_proba(X)[:, 1]

print("model refit on the same client-grouped train fold as the capstone -- consistent, not re-tuned")


model refit on the same client-grouped train fold as the capstone -- consistent, not re-tuned


In [2]:
expected_ctr_for_tier = df.groupby("position_tier")["ctr"].transform("mean")

REVIEW_THRESHOLD = df["model_score"].quantile(0.90)  # top 10% -> one editorial sprint's worth
df["action"] = np.where(df["model_score"] >= REVIEW_THRESHOLD, "review_declining_risk", "monitor")

def assign_reason(row):
    if row["impressions_90d"] < 10:
        return "near_zero_signal"
    if row["ctr"] < expected_ctr_for_tier.loc[row.name] * 0.5:
        return "ctr_below_position_expectation"
    if row["content_age_days"] < 90:
        return "young_and_volatile"
    if row["days_with_impressions"] < df["days_with_impressions"].quantile(0.25):
        return "low_active_days"
    return "model_flagged_review"

flagged = df["action"] == "review_declining_risk"
df.loc[flagged, "reason_code"] = df.loc[flagged].apply(assign_reason, axis=1)
df.loc[~flagged, "reason_code"] = "not_flagged"

print("flagged for review:", flagged.sum(), "of", len(df))
df.loc[flagged, "reason_code"].value_counts()


flagged for review: 3000 of 30000


reason_code
ctr_below_position_expectation    2384
model_flagged_review               612
near_zero_signal                     3
low_active_days                      1
Name: count, dtype: int64

In [3]:
# archetype -> action mapping (independent of the model score -- a simple, transparent default)
median_impr = df["impressions_90d"].median()

def archetype(row):
    if row["trend_direction"] == "new":
        return "New / Unproven"
    if row["trend_direction"] == "up":
        return "Growing & Visible" if row["impressions_90d"] >= median_impr else "Growing & Low-Visibility"
    if row["trend_direction"] in ("stable", "flat"):
        return "Stable & Visible" if row["impressions_90d"] >= median_impr else "Stable & Low-Visibility"
    if row["trend_direction"] == "down":
        return "Declining & Visible" if row["impressions_90d"] >= median_impr else "Declining & Low-Visibility"
    return "Other"

df["archetype"] = df.apply(archetype, axis=1)

archetype_action = {
    "Growing & Visible": "Protect -- leave as-is; candidate for an internal best-practice example",
    "Growing & Low-Visibility": "Promote -- good trend, under-exposed; consider internal linking",
    "Stable & Visible": "Monitor -- healthy, periodic check-in only",
    "Stable & Low-Visibility": "Low-priority monitor",
    "Declining & Visible": "Review -- this is the core reason-coded queue above",
    "Declining & Low-Visibility": "Deprioritize / prune candidate -- low cost of inaction",
    "New / Unproven": "Watch -- too early to assess, exclude from the action queue entirely",
}

counts = df["archetype"].value_counts()
archetype_table = pd.DataFrame({
    "archetype": counts.index,
    "n_pages": counts.values,
    "default_action": [archetype_action[a] for a in counts.index],
})
archetype_table


,archetype,n_pages,default_action
0,Declining & Visible,8914,Review -- this is the core reason-coded queue ...
1,Declining & Low-Visibility,7348,Deprioritize / prune candidate -- low cost of ...
2,Stable & Visible,3992,"Monitor -- healthy, periodic check-in only"
3,Stable & Low-Visibility,3122,Low-priority monitor
4,Growing & Low-Visibility,2374,"Promote -- good trend, under-exposed; consider..."
5,New / Unproven,2236,"Watch -- too early to assess, exclude from the..."
6,Growing & Visible,2014,Protect -- leave as-is; candidate for an inter...


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who uses this:** a content or SEO team deciding which pages to review in a limited weekly or
monthly editorial sprint — a human prioritization aid, not an automated content or publishing
system.

**What it's for:** ranking a large content library by observed decline risk and current-state
signals, with a plain-English reason attached, so a reviewer can quickly judge whether the
model's reasoning holds up for that specific page.

**Where it stops being valid:**
- Built on a 30,000-row anonymized starter sample from one portfolio snapshot — not the full
  warehouse, and not validated on any other content ecosystem or vertical.
- A **current-state diagnostic**, not a forecast: it describes what a page's present profile
  resembles, not what will happen to it next, and it makes no causal claim about *why* a page is
  declining or that any fix will reverse it.
- Trained on one time snapshot; untested across seasons, algorithm updates, or portfolio
  composition changes — see Section 4 for what would signal it's gone stale.
- All claims here are observed, measured, and directional — never causal, per the claim ladder.


In [4]:
honest_receipts = {
    "precision_at_K_grouped_holdout": 0.689,
    "roc_auc_grouped_holdout": 0.632,
    "baseline_rule_precision_at_K": 0.596,
    "test_fold_base_rate": 0.559,
    "portfolio_base_rate": round(float(df["is_declining"].mean()), 3),
    "n_pages_scored": int(len(df)),
    "n_clients": int(df["client_id"].nunique()),
}
print("receipts this playbook is built on (from the capstone's held-out evaluation):")
for k, v in honest_receipts.items():
    print(f"  {k}: {v}")


receipts this playbook is built on (from the capstone's held-out evaluation):
  precision_at_K_grouped_holdout: 0.689
  roc_auc_grouped_holdout: 0.632
  baseline_rule_precision_at_K: 0.596
  test_fold_base_rate: 0.559
  portfolio_base_rate: 0.542
  n_pages_scored: 30000
  n_clients: 32


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**What a reviewer must check before acting on any flagged row:**
1. Does the reason code point to something actually fixable? (`ctr_below_position_expectation`
   can also mean a featured snippet or answer box is absorbing clicks — no title/meta rewrite
   fixes that.)
2. For `near_zero_signal` or `young_and_volatile` rows: confirm the page isn't simply newly
   published rather than genuinely declining — Week 4's own signal audit found staleness runs
   *opposite* to intuition here, so age-based assumptions deserve extra scrutiny.
3. Does the page's current topic still match business priorities? A page can correctly decline
   because the business intentionally deprioritized that topic — that's not a model failure.

**What should NEVER be automated (the no-go list):**
- Auto-publishing content or metadata edits based on a reason code
- Auto-redirecting, merging, or deleting pages from the ranked queue
- Auto-adjusting ad spend or budget tied to a page's score
- Using an individual page's score as a performance metric for whoever wrote it
- Treating any single flag as proof the content or SEO strategy "failed"
- Applying this playbook's rules or thresholds to a different portfolio or vertical without
  re-running the signal audit first (per Week 6's methodology review of FlyRank's own paper)


In [5]:
no_go_list = [
    "auto-publish content or metadata edits from a reason code",
    "auto-redirect, merge, or delete pages from the ranked queue",
    "auto-adjust ad spend or budget tied to a page's score",
    "use an individual page's score as an author performance metric",
    "treat a single flag as proof the content/SEO strategy failed",
    "reuse these thresholds on a different portfolio without re-running the signal audit",
]
for item in no_go_list:
    print("NO-GO:", item)


NO-GO: auto-publish content or metadata edits from a reason code
NO-GO: auto-redirect, merge, or delete pages from the ranked queue
NO-GO: auto-adjust ad spend or budget tied to a page's score
NO-GO: use an individual page's score as an author performance metric
NO-GO: treat a single flag as proof the content/SEO strategy failed
NO-GO: reuse these thresholds on a different portfolio without re-running the signal audit


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Signals the recommendations have gone stale:**
1. **`model_flagged_review` share grows** — this is the catch-all reason code for rows the
   explainable rules can't pin down; a rising share means the model is keying on patterns the
   simple rules no longer capture, worth investigating.
2. **Precision@K on a fresh holdout drops meaningfully below 0.689** (today's honest baseline) —
   direct evidence the ranking no longer works as well as it did.
3. **Portfolio base rate shifts a lot** from today's 0.542 — a big change in what "normal"
   decline looks like (new client mix, algorithm update) means the whole frame may need
   rebuilding, not just retraining.
4. **The age-tier signal reverses again** — if older content starts declining *more* than
   newer content (the opposite of Week 4's finding), the underlying dynamics have likely
   changed and the rule set should be re-audited, not just the model.

**Cadence:** recompute quarterly, given the trailing-90-day feature windows already blur month
to month — or immediately, off-schedule, if any trigger above fires.


In [6]:
retrain_triggers = {
    "reference_precision_at_K": 0.689,
    "reference_base_rate": round(float(df["is_declining"].mean()), 3),
    "reference_model_flagged_review_share": round(float((df.loc[flagged, "reason_code"] == "model_flagged_review").mean()), 3),
    "recommended_recompute_cadence_days": 90,
}
print("stored as the comparison point for future runs:")
for k, v in retrain_triggers.items():
    print(f"  {k}: {v}")


stored as the comparison point for future runs:
  reference_precision_at_K: 0.689
  reference_base_rate: 0.542
  reference_model_flagged_review_share: 0.204
  recommended_recompute_cadence_days: 90


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Ranked queue → `work/outputs/` (gitignored, regenerated on every run). Metrics/receipts JSON →
also `work/outputs/`, but committed per the CI policy, since it's small and non-sensitive.
Cost/value framing: reviewing the top 10% by score catches a confirmed decliner 68.9% of the
time on held-out data (vs. 54.2% from reviewing at random) — meaningfully fewer wasted reviews
per real catch, for a fixed amount of editorial time.


In [7]:
import os
os.makedirs("../outputs", exist_ok=True)
os.makedirs("../figures", exist_ok=True)

queue_cols = ["content_id", "content_type", "archetype", "action", "reason_code", "model_score",
              "position_tier", "avg_position", "ctr", "impressions_90d", "content_age_days"]
ranked_queue = df.sort_values("model_score", ascending=False)[queue_cols]
ranked_queue.to_csv("../outputs/w07_ranked_playbook.csv", index=False)
print("written (gitignored): work/outputs/w07_ranked_playbook.csv --", len(ranked_queue), "rows")

import json as jsonlib
metrics_receipt = {
    **honest_receipts,
    **retrain_triggers,
    "reason_code_counts": df.loc[flagged, "reason_code"].value_counts().to_dict(),
    "archetype_counts": df["archetype"].value_counts().to_dict(),
}
with open("../outputs/w07_playbook_metrics.json", "w") as f:
    jsonlib.dump(metrics_receipt, f, indent=2, default=str)
print("written (committed): work/outputs/w07_playbook_metrics.json")


written (gitignored): work/outputs/w07_ranked_playbook.csv -- 30000 rows
written (committed): work/outputs/w07_playbook_metrics.json
